In [ ]:
# %load_ext autoreload
# %autoreload 2

import sys
sys.path.append('/home/galk/LanguageDynamics/src') 
from data_generation import *
from config import TinyLMConfig, TinyAutoencoderConfig, TinyKoopmanAutoencoderConfig, TrainingConfig, ExperimentConfig
from models import TinyLlamaTransformer
from models import TransformerAutoencoder
import random
import numpy as np
from tqdm import tqdm

import os
import math
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
# from datasets import load_dataset
from datetime import datetime

from importlib import reload

In [ ]:
### Hyperparameters
E = 1
D = 1
N = 4
M = 2
G = 8
L_E = [1 for _ in range(E)]
L_D = [1 for _ in range(D)]
L_N = [3, 4, 5, 6]
L_M = [1 for _ in range(M)]

special_tokens = []

G_MIN = len(special_tokens) + E + D + M  # special tokens, plus other tokens
G_MAX = G_MIN + G - 1

memory_limit = float("inf")

### Sample noise trajectories
noise_list = []
for n_i in range(N):
    noise_list.append(random.sample(range(G_MIN, G_MAX+1), L_N[n_i]))

vocab = special_tokens + [f"E{e+1}" for e in range(E)] + [f"D{d+1}" for d in range(D)] + [f"M{m+1}" for m in range(M)] + [f"G{g+1}" for g in range(G)]
token2id = {tok: idx for idx, tok in enumerate(vocab)}
id2token = {idx: tok for tok, idx in token2id.items()}
vocab_size = len(vocab)

# Non-Terminal (NT) vocab
NT_vocab = special_tokens + [f"E{e+1}" for e in range(E)] + [f"D{d+1}" for d in range(D)] + [f"M{m+1}" for m in range(M)] + [f"N{n+1}" for n in range(N)]
NT_token2id = {tok: idx for idx, tok in enumerate(NT_vocab)}
NT_id2token = {idx: tok for tok, idx in NT_token2id.items()}
NT_vocab_size = len(NT_vocab)

# Initialize NT to T transitions
NT_to_T = {token: token if token not in [f"N{n+1}" for n in range(N)] else [id2token[id] for id in noise_list[[f"N{n+1}" for n in range(N)].index(token)]] for token in NT_vocab }

### Initialize Transition matrices
# P_transitions = np.zeros((E+1, NT_vocab_size, NT_vocab_size))  # [states, source, target]
P_transitions = np.zeros((1, NT_vocab_size, NT_vocab_size))  # [states, source, target]

# zero mode - no memory
# P_transitions[0, NT_token2id['<EOS>'], NT_token2id['<BOS>']] = 1
# P_transitions[0, NT_token2id.<BOS>'], NT_token2id.E1']] = 1/(2*N)
# P_transitions[0, NT_token2id['<BOS>'], NT_token2id['N1']:NT_token2id[f'N{N}']+1] = 1/N
# P_transitions[0, NT_token2id['M1']:NT_token2id[f'M{M}']+1, NT_token2id['E1']:NT_token2id[f'E{E}']+1] = 1/(2*N)
P_transitions[0, NT_token2id['E1']:NT_token2id[f'E{E}']+1, NT_token2id['M1']:NT_token2id[f'M{M}']+1] = 1/M
P_transitions[0, NT_token2id['M1']:NT_token2id[f'M{M}']+1, NT_token2id['N1']:NT_token2id[f'N{N}']+1] = 1/(N)
# P_transitions[0, NT_token2id['M1']:NT_token2id[f'M{M}']+1, NT_token2id['<EOS>']] = 0.2
P_transitions[0, NT_token2id['N1']:NT_token2id[f'N{N}']+1, NT_token2id['N1']:NT_token2id[f'N{N}']+1] = 1/(N)
P_transitions[0, NT_token2id['N1']:NT_token2id[f'N{N}']+1, NT_token2id['E1']:NT_token2id[f'E{E}']+1] = 1/(N)
P_transitions[0, NT_token2id['N1']:NT_token2id[f'N{N}']+1, NT_token2id['D1']:NT_token2id[f'D{D}']+1] = 1/(N)
# P_transitions[0, NT_token2id['N1']:NT_token2id[f'N{N}']+1, NT_token2id['<EOS>']] = 1/(2*N)
np.fill_diagonal(P_transitions[0], 0)

# memory mode
# P_transitions[1, NT_token2id['D1']:NT_token2id[f'E{E}']+1, NT_token2id['M1']:NT_token2id[f'M{M}']+1] = 1/M  ## THIS WILL BE DECIDED BY THE CONTEXT (which M was memorized)
# P_transitions[1, NT_token2id['E1']:NT_token2id[f'E{E}']+1, NT_token2id['M1']:NT_token2id[f'M{M}']+1] = 1/M
# P_transitions[1, NT_token2id['M1']:NT_token2id[f'M{M}']+1, NT_token2id['N1']:NT_token2id[f'N{N}']+1] = 1/(N)
# P_transitions[1, NT_token2id['N1']:NT_token2id[f'N{N}']+1, NT_token2id['N1']:NT_token2id[f'N{N}']+1] = 1/(N)
# P_transitions[1, NT_token2id['N1']:NT_token2id[f'N{N}']+1, NT_token2id['D1']:NT_token2id[f'D{D}']+1] = 2/(N)
# np.fill_diagonal(P_transitions[1], 0)


# Normalize Matrices
P_transitions = P_transitions / P_transitions.sum(axis=-1, keepdims=True) # normalize rows to sum to 1
P_transitions = np.nan_to_num(P_transitions) # replace NaNs with 0


In [ ]:
model_config = TinyLMConfig(
    n_layers=4,
    n_heads=4,
    embed_dim=32,
    ffn_dim=256,
    context_window=32,
    vocab=NT_vocab,
    dropout_self_attention=0.05,
    dropout_embed=0.03,
    dropout_residual=0.03
)

training_config = TrainingConfig(
    lr=1e-4,
    batch_size=256,
    grad_clipping=True
)

experiment_config = ExperimentConfig(
    epochs=500,
    # checkpoint_path=None,
    checkpoint_path="/home/galk/LanguageDynamics/models/linguistic_flip_flop/tiny_LM_dyck2_layers_4_embed_32_ffn_dim_256_context_window_32_date_110925_1816_trial_1/ckpt_epoch2.pt",
    save_every=1,
    device_index=0,
    model_config=model_config,
    training_config=training_config
)

pad_id = 100

# TODO: move the model save prefix to the ExperimentConfig initialization
current_date_ddmmyy = datetime.now().strftime('%d%m%y_%H%M')
if model_config.mode == 'AE':
    experiment_config.model_save_prefix = f"tiny_{model_config.mode}_dyck2_layers_{model_config.n_layers}_embed_{model_config.embed_dim}_ffn_dim_{model_config.ffn_dim}_context_window_{model_config.context_window}_latent_{model_config.latent_dim}_n_latents_{model_config.n_latents}_maxdepth_{model_config.max_depth}_maxlen_{model_config.max_length}_date_{current_date_ddmmyy}"
if model_config.mode == 'KAE':
    experiment_config.model_save_prefix = f"tiny_{model_config.mode}_dyck2_layers_{model_config.n_layers}_embed_{model_config.embed_dim}_ffn_dim_{model_config.ffn_dim}_context_window_{model_config.context_window}_latent_{model_config.latent_dim}_n_latents_{model_config.n_latents}_n_diagonals_{model_config.n_diagonals}_maxdepth_{model_config.max_depth}_maxlen_{model_config.max_length}_date_{current_date_ddmmyy}"
elif model_config.mode == 'LM' or model_config.mode == 'RLM':
    # TODO: add memory generation parameters to name
    experiment_config.model_save_prefix = f"tiny_{model_config.mode}_dyck2_layers_{model_config.n_layers}_embed_{model_config.embed_dim}_ffn_dim_{model_config.ffn_dim}_context_window_{model_config.context_window}_date_{current_date_ddmmyy}"

print(f"Using device: {experiment_config.device}")

In [ ]:
context_window = experiment_config.model_config.context_window
n_train = 1
n_train_windows = 100000
max_steps = n_train_windows * (context_window + 1)
val_ratio = 0.01
train_dataset, train_dataset_NT, train_seen = generate_tiny_memory_dataset(n_samples=n_train, P_transitions=P_transitions, NT_vocab=NT_vocab,NT_token2id=NT_token2id, NT_to_T=NT_to_T, num_steps=max_steps, memory_limit=memory_limit)
val_dataset, val_dataset_NT, val_seen = generate_tiny_memory_dataset(n_samples=n_train, seen=train_seen, P_transitions=P_transitions, NT_vocab=NT_vocab,NT_token2id=NT_token2id, NT_to_T=NT_to_T, num_steps=int(max_steps*val_ratio), memory_limit=memory_limit)
train_blocks = prepare_blocks(train_dataset_NT, NT_token2id, context_window)
val_blocks = prepare_blocks(val_dataset_NT, NT_token2id, context_window)

In [ ]:
print(train_blocks.shape)
print(val_blocks.shape)

In [ ]:
# Remote (SSH) - Linux path
models_path = '/home/galk/LanguageDynamics/models/linguistic_flip_flop'

In [ ]:
# Create datasets
train_dataset = TokensDataset(train_blocks)
val_dataset = TokensDataset(val_blocks)

# Create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=experiment_config.training_config.batch_size,
    shuffle=True,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=experiment_config.training_config.batch_size,
    shuffle=False,  # No need to shuffle validation data
    drop_last=True
)

# Verify the split
print("\nDataLoader Info:")
print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")

In [ ]:
# --------- 7. Training --------- WITH CHECKPOINT LOADING 26/06/25
import matplotlib.pyplot as plt
from datetime import datetime
import time
from pathlib import Path

def calculate_accuracy(logits, targets, pad_id):
    # Flatten predictions and targets
    pred = logits.argmax(dim=-1).view(-1)
    true = targets.view(-1)
    # Create mask to ignore padding tokens
    mask = (true != pad_id)
    # Calculate accuracy only on non-pad tokens
    correct = (pred[mask] == true[mask]).float().sum()
    total = mask.sum()
    return (correct / total).item() if total > 0 else 0

class MetricsTracker:
    def __init__(self):
        self.train_losses = []
        self.train_accuracies = []
        self.val_losses = []
        self.val_accuracies = []
        self.epochs = []

    def load_from_checkpoint(self, metrics_dict):
        """Load metrics from a checkpoint dictionary"""
        self.train_losses = metrics_dict['train_losses']
        self.train_accuracies = metrics_dict['train_accuracies']
        self.val_losses = metrics_dict['val_losses']
        self.val_accuracies = metrics_dict['val_accuracies']
        self.epochs = list(range(1, len(self.train_losses) + 1))

    def plot_metrics(self, save_dir=None):
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

        # Plot losses
        ax1.plot(self.epochs, self.train_losses, label='Train Loss')
        ax1.plot(self.epochs, self.val_losses, label='Val Loss')
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('Loss')
        ax1.set_title('Training and Validation Loss')
        ax1.legend()
        ax1.grid(True)

        # Plot accuracies
        ax2.plot(self.epochs, self.train_accuracies, label='Train Accuracy')
        ax2.plot(self.epochs, self.val_accuracies, label='Val Accuracy')
        ax2.set_xlabel('Epoch')
        ax2.set_ylabel('Accuracy')
        ax2.set_title('Training and Validation Accuracy')
        ax2.legend()
        ax2.grid(True)

        plt.tight_layout()

        if save_dir:
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            plt.savefig(f"{save_dir}/metrics_{timestamp}.png")

        plt.show()

def load_checkpoint(checkpoint_path, model, device, optimizer=None):
    """
    Load model and optimizer state from a checkpoint
    Returns the starting epoch and loaded metrics
    """
    print(f"Loading checkpoint from {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location=device)

    # Load model state
    model.load_state_dict(checkpoint['model_state_dict'])

    # Load optimizer state if provided
    if optimizer is not None:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

    # Verify config matches
    saved_config = checkpoint['config']
    for key in ['vocab_size', 'embed_dim', 'n_layers', 'n_heads', 'ffn_dim', 'context_window']:
        if CONFIG[key] != saved_config[key]:
            raise ValueError(f"Checkpoint config mismatch for {key}: "
                           f"current={CONFIG[key]}, saved={saved_config[key]}")

    starting_epoch = checkpoint['epoch']
    return starting_epoch, checkpoint['metrics']

n_trials = 1
device = experiment_config.device
for trial in range(n_trials):
    # Initialize model
    if experiment_config.model_config.mode == 'LM':
        model = TinyLlamaTransformer(
            experiment_config.model_config
        ).to(device)

    elif experiment_config.model_config.mode == 'AE':
        model = TransformerAutoencoder(
            experiment_config.model_config
        ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=experiment_config.training_config.lr)
    criterion = nn.CrossEntropyLoss(ignore_index=pad_id)
    metrics = MetricsTracker()

    # Create save directory
    save_dir = Path(os.path.join(models_path, experiment_config.model_save_prefix + f"_trial_{trial+1}"))
    save_dir.mkdir(exist_ok=True, parents=True)

    # Load checkpoint if specified
    starting_epoch = 0
    if experiment_config.checkpoint_path:
        try:
            starting_epoch, saved_metrics = load_checkpoint(
                experiment_config.checkpoint_path,
                model,
                device,
                optimizer
            )
            for param_group in optimizer.param_groups:
                param_group['lr'] = experiment_config.training_config.lr
            metrics.load_from_checkpoint(saved_metrics)
            print(f"Resuming training from epoch {starting_epoch}")
        except Exception as e:
            print(f"Error loading checkpoint: {e}")
            print("Starting training from scratch")
            starting_epoch = 0

    print("Starting training loop...")
    global_start = time.time()

    def validate(model, val_loader, criterion, model_config, device):
        model.eval()
        total_loss = 0
        total_acc = 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                if model_config.mode == 'LM':
                    logits = model(x)
                elif model_config == 'AE':
                    logits, _, _, _ = model(x, decoding_seed=x[:, :-1])
                loss = criterion(logits.view(-1, len(model_config.vocab)), y.view(-1))
                acc = calculate_accuracy(logits, y, pad_id)
                total_loss += loss.item()
                total_acc += acc
        return total_loss / len(val_loader), total_acc / len(val_loader)

    # Training loop
    for epoch in range(starting_epoch, experiment_config.epochs):
        model.train()
        epoch_loss = 0.0
        epoch_acc = 0.0
        start = time.time()

        # Training phase
        for batch, (x, y) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1} Training")):
            x = x.to(device)
            y = y.to(device)

            if experiment_config.model_config.mode == 'LM':
                logits = model(x)
            elif experiment_config.model_config.mode == 'AE':
                logits, _, _, _ = model(x, decoding_seed=x[:,:-1])
            loss = criterion(logits.view(-1, len(experiment_config.model_config.vocab)), y.view(-1))
            acc = calculate_accuracy(logits, y, pad_id)

            optimizer.zero_grad()
            loss.backward()

            # Optional: Gradient clipping
            if experiment_config.training_config.grad_clipping:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            optimizer.step()

            epoch_loss += loss.item()
            epoch_acc += acc

        # Calculate training metrics
        train_loss = epoch_loss / len(train_loader)
        train_acc = epoch_acc / len(train_loader)

        # Validation phase
        val_loss, val_acc = validate(model, val_loader, criterion, experiment_config.model_config, device)

        # Update metrics
        metrics.epochs.append(epoch + 1)
        metrics.train_losses.append(train_loss)
        metrics.train_accuracies.append(train_acc)
        metrics.val_losses.append(val_loss)
        metrics.val_accuracies.append(val_acc)

        end = time.time()

        # Print metrics
        print(f"\n[Epoch {epoch+1}]")
        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
        print(f"Epoch time: {end-start:.1f}s | Total time: {end-global_start:.1f}s")

        # Plot metrics
        metrics.plot_metrics(save_dir)

        # Save checkpoint
        if (epoch+1) % experiment_config.save_every == 0 or (epoch+1) == experiment_config.epochs:
            # ckpt_path = save_dir / f"{CONFIG['model_save_prefix']}_trial_{trial+1}_epoch{epoch+1}.pt"
            ckpt_path = save_dir / f"ckpt_epoch{epoch+1}.pt"
            torch.save({
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'config': experiment_config,
                'epoch': epoch+1,
                'metrics': {
                    'train_losses': metrics.train_losses,
                    'train_accuracies': metrics.train_accuracies,
                    'val_losses': metrics.val_losses,
                    'val_accuracies': metrics.val_accuracies
                }
            }, ckpt_path)
            print(f"Saved checkpoint: {ckpt_path}")

    print("Training complete!")

In [ ]:
# Loading a saved model
ckpt_path = experiment_config.checkpoint_path
# ckpt_path = "/content/tiny_transformer_dyck2_layers_2_embed_32_context_window_64_weight_tied_maxdepth_20_max_len_50_date_29_07_25_trial_1/model_epoch15.pt"
checkpoint = torch.load(ckpt_path, map_location=torch.device(experiment_config.device), weights_only=False)

# if 'mode' not in checkpoint['config']:
    # checkpoint['config']['mode'] = "LM"  # Default to LM if not specified

if checkpoint['config'].model_config.mode == 'LM':
    model = TinyLlamaTransformer(
        checkpoint['config'].model_config
    ).to(experiment_config.device)

elif checkpoint['config'].model_config.mode == 'RLM':
        model = TinyLlamaRawTransformer(
            embed_dim=CONFIG['embed_dim'],
            n_layers=CONFIG['n_layers'],
            n_heads=CONFIG['n_heads'],
            ffn_dim=CONFIG['ffn_dim'],
            context_window=CONFIG['context_window']
        ).to(CONFIG['device'])

elif checkpoint['config'].model_config.mode == 'AE':
    model = TransformerAutoencoder(
            vocab_size=checkpoint['config']['vocab_size'],
            embed_dim=checkpoint['config']['embed_dim'],
            latent_dim=checkpoint['config']['latent_dim'],
            n_layers=checkpoint['config']['n_layers'],
            n_heads=checkpoint['config']['n_heads'],
            ffn_dim=checkpoint['config']['ffn_dim'],
            context_window=checkpoint['config']['context_window'],
            cls_id=cls_id,
            sos_id=sos_id,
            n_latents=checkpoint['config']['n_latents']
        ).to(CONFIG['device'])
elif checkpoint['config'].model_config.mode == 'KAE':
    model = TransformerKoopmanAutoencoder(
            vocab_size=CONFIG['vocab_size'],
            embed_dim=CONFIG['embed_dim'],
            latent_dim=CONFIG['latent_dim'],
            n_layers=CONFIG['n_layers'],
            n_heads=CONFIG['n_heads'],
            ffn_dim=CONFIG['ffn_dim'],
            context_window=CONFIG['context_window'],
            cls_id=cls_id,
            sos_id=sos_id,
            n_latents=CONFIG['n_latents'],
            n_diagonals=CONFIG['n_diagonals']
        ).to(CONFIG['device'])


model_state = checkpoint['model_state_dict']
model.load_state_dict(model_state)

In [ ]:
prompt = ['E1', "M1", "N1"]

max_new_tokens = 80
temperature = 1
top_k = 10

generated, probs = generate_from_model(
    model=model,
    token2id=NT_token2id,
    id2token=NT_id2token,
    prompt=prompt,
    max_new_tokens=max_new_tokens,
    temperature=temperature,
    top_k=top_k,
    device=experiment_config.device
)

print(generated)

In [ ]:

fig = plot_generation_probabilities(
    probabilities=probs,
    sequence=generated[len(prompt):],
    token2id=NT_token2id,
    id2token=NT_id2token,
    tokens_to_highlight=None,
    figsize=(12, 8)
)